# 1. SneakPeek
Having a SneakPeek at the files before processing for transmormations. Never load a file you havent look at. The `read_files` function lets you query a file in volume directly, like it was already a table.

In [0]:
%sql
select * 
from read_files(
    '/Volumes/shopstream/core/raw/orders_2026_h1.csv',
    format => 'csv',
    header => true
)limit 10;

order_line_id,order_id,customer_id,product_id,quantity,unit_price,order_ts,status,coupon_code,_rescued_data
L0000001,O000001,C00628,P0123,1,106.28,2026-02-20T07:14:43.000Z,completed,FESTIVE20,null
L0000002,O000001,C00628,P0096,1,267.14,2026-02-20T07:14:43.000Z,completed,WELCOME15,null
L0000003,O000002,C00141,P0057,2,22.09,2026-02-07T07:02:26.000Z,completed,null,null
L0000004,O000002,C00141,P0017,1,53.8,2026-02-07T07:02:26.000Z,completed,FESTIVE20,null
L0000005,O000002,C00141,P0126,1,107.67,2026-02-07T07:02:26.000Z,completed,FESTIVE20,null
L0000006,O000002,C00141,P0110,1,27.62,2026-02-07T07:02:26.000Z,completed,null,null
L0000007,O000003,C00845,P0165,2,63.33,2026-04-24T10:25:39.000Z,cancelled,null,null
L0000008,O000004,C00463,P0011,1,64.02,2026-04-27T01:56:15.000Z,returned,null,null
L0000009,O000004,C00463,P0113,1,319.88,2026-04-27T01:56:15.000Z,returned,null,null
L0000010,O000004,C00463,P0074,1,156.41,2026-04-27T01:56:15.000Z,returned,WELCOME15,null


# 2. Load Orders with COPY INTO
-   You could load those files with plain Create Table As Select (CTAS). But production pipelines almost never do that for file ingestion, because CTAS reloads everything everytime. The production tool is COPY INTO and the difference matters!
- CTAS rebuilds the table from whatever it reads. if we run this command twice then either data is duplicated or wipe and reload
- COPY INTO tracks which file it already loaded. Run it twice, and second run loads nothing. That property is called Idempotency, and it is what lets you re-run a failed pipeline without fear.

In [0]:
%sql
-- Bronze Orders: create once, then load with copy into (Idempotent)

create table if not exists shopstream.core.bronze_orders;

copy into shopstream.core.bronze_orders
from '/Volumes/shopstream/core/raw/orders_2026_h1.csv'
fileformat = csv
format_options(
    'header' = 'true',
    'inferSchema' = 'true',
    'mergeSchema' = 'true'
)
copy_options(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


# Load the remaining two dim tables

In [0]:
%sql
create table if not exists shopstream.core.bronze_customers;

copy into shopstream.core.bronze_customers
from '/Volumes/shopstream/core/raw/customers.csv'
fileformat = csv
format_options(
    'header' = 'true',
    'inferSchema' = 'true',
    'mergeSchema' = 'true'
)
copy_options(
    'mergeSchema' = 'true'
);



create table if not exists shopstream.core.bronze_products;

copy into shopstream.core.bronze_products
from '/Volumes/shopstream/core/raw/products.csv'
fileformat = csv
format_options(
    'header' = 'true',
    'inferSchema' = 'true',
    'mergeSchema' = 'true'
)
copy_options(
    'mergeSchema' = 'true'
);


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
197,197,0


# Verify the Layer

In [0]:
%sql
select 'bronze_customers' as table_name, count(*) as row_count from shopstream.core.bronze_customers
union all
select 'bronze_orders', count(*) from shopstream.core.bronze_orders
union all
select 'bronze_products', count(*) from shopstream.core.bronze_products;

table_name,row_count
bronze_customers,1000
bronze_orders,13717
bronze_products,197
